In [1]:
!pip install transformers==4.44.2 joblib==1.4.2 scikit-learn==1.6.0 numpy==1.26.4 pandas==2.2.3 scipy==1.13.1 seaborn==0.13.2 tqdm==4.66.5 lightgbm==4.5.0 xgboost==2.1.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 364.6 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 20.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 14.4 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 34.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.9/153.9 MB 10.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 74.1 MB/s eta 0:00:00:00:01
  Attempting uninstall: tqdm
    Found existing installation: t

In [2]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
import torch
from torch.utils.data import DataLoader, Dataset


# Load datasets
train_df = pd.read_csv('/kaggle/input/pampa-dataset/Train.csv')
train_df = train_df[['ID', 'SMILES', 'Permeability']]
test_df = pd.read_csv('/kaggle/input/pampa-dataset/Test.csv')
test_df = test_df[['ID', 'SMILES', 'Permeability']]

In [3]:
tokenizer = AutoTokenizer.from_pretrained("seyonec/PubChem10M_SMILES_BPE_450k")
model = AutoModelForSequenceClassification.from_pretrained('seyonec/PubChem10M_SMILES_BPE_450k', num_labels=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/515 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/336M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at seyonec/PubChem10M_SMILES_BPE_450k and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# Custom dataset class
class SMILESDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=325):
        self.tokenizer = tokenizer
        self.dataframe = dataframe
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        smiles = self.dataframe.iloc[idx]['SMILES']
        permeability = self.dataframe.iloc[idx]['Permeability']
        inputs = self.tokenizer(smiles, return_tensors='pt', padding="max_length", truncation=True, max_length=self.max_length)
        
        input_ids = inputs['input_ids'].squeeze(0)  # Shape: (sequence_length,)
        attention_mask = inputs['attention_mask'].squeeze(0)  # Shape: (sequence_length,)
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(permeability, dtype=torch.float)
        }

In [5]:
# datasets
train_dataset = SMILESDataset(train_df, tokenizer)
test_dataset = SMILESDataset(test_df, tokenizer)
batch_size = 16
# data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [6]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_epochs = 20

In [7]:
# Training loop
from tqdm import tqdm
for epoch in range(num_epochs):
    print(f"Entered Epoch {epoch + 1}")
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f'Training Epoch {epoch + 1}/{num_epochs}', unit='batch'):
        optimizer.zero_grad()

        # Move all batch tensors to device
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch["labels"].unsqueeze(1)  # still shape: (batch_size, 1)

        # Forward pass
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=labels
        )
        loss = outputs.loss
        train_loss += loss.item()

        # Backprop and optimizer step
        loss.backward()
        optimizer.step()

    avg_train_loss = train_loss / len(train_loader)
    print(f'Epoch {epoch + 1}/{num_epochs} - Train Loss: {avg_train_loss:.4f}')

Entered Epoch 1


Training Epoch 1/20: 100%|██████████| 348/348 [02:56<00:00,  1.97batch/s]


Epoch 1/20 - Train Loss: 0.7793
Entered Epoch 2


Training Epoch 2/20: 100%|██████████| 348/348 [03:06<00:00,  1.86batch/s]


Epoch 2/20 - Train Loss: 0.3998
Entered Epoch 3


...

Epoch 18/20 - Train Loss: 0.1396
Entered Epoch 19


Training Epoch 19/20: 100%|██████████| 348/348 [03:06<00:00,  1.86batch/s]


Epoch 19/20 - Train Loss: 0.1346
Entered Epoch 20


Training Epoch 20/20: 100%|██████████| 348/348 [03:06<00:00,  1.86batch/s]

Epoch 20/20 - Train Loss: 0.1353


In [8]:
# Saving the model after training
model_name = 'PubChem10M_SMILES_BPE_450k_model_1_pampa'
model_save_path = f'/kaggle/working/{model_name}'
os.makedirs(model_save_path, exist_ok=True)

tokenizer.save_pretrained(model_save_path)
model.save_pretrained(model_save_path)

print(f'Model and tokenizer saved to {model_save_path}')

Model and tokenizer saved to /kaggle/working/PubChem10M_SMILES_BPE_450k_model_1_pampa


In [9]:
from scipy.stats import pearsonr, spearmanr

model.eval()
test_loss = 0
test_true_labels = []
predictions = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing', unit='batch'):
      
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].unsqueeze(1).to(device).float()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        test_loss += loss.item()

        test_true_labels.extend(labels.cpu().numpy())
        preds = outputs.logits.squeeze().cpu().numpy()  
        predictions.extend(preds)

# Final test loss
avg_test_loss = test_loss / len(test_loader)
print(f'Test Loss: {avg_test_loss:.4f}')

test_true_labels = np.array(test_true_labels).flatten()
predictions = np.array(predictions)
print(test_true_labels.shape)
print(predictions.shape)

# Performance metrics
mse = mean_squared_error(test_true_labels, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_true_labels, predictions)
r2 = r2_score(test_true_labels, predictions)
PCC,_ = pearsonr(test_true_labels, predictions)
SCC,_ = spearmanr(test_true_labels, predictions)

# Print performance metrics
print(f'Mean Squared Error: {mse:.4f}')
print(f'Root Mean Squared Error: {rmse:.4f}')
print(f'Mean Absolute Error: {mae:.4f}')
print(f'R^2 Score: {r2:.4f}')
print(f'Pearson Correlation Coefficient: {PCC:.4f}')
print(f'Spearman Correlation Coefficient: {SCC:.4f}')

# Print hyperparameters
print("Hyperparameters:")
print(f"Learning Rate: {5e-5}")
print(f"Batch Size: 16")
print(f"Epochs: {num_epochs}")

Testing: 100%|██████████| 87/87 [00:16<00:00,  5.31batch/s]


Test Loss: 0.2809
(1392,)
(1392,)
Mean Squared Error: 0.2809
Root Mean Squared Error: 0.5300
Mean Absolute Error: 0.3758
R^2 Score: 0.5582
Pearson Correlation Coefficient: 0.7620
Spearman Correlation Coefficient: 0.7440
Hyperparameters:
Learning Rate: 5e-05
Batch Size: 16
Epochs: 20


In [10]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model_name = 'PubChem10M_SMILES_BPE_450k_model_1_pampa'
model_save_path = f'/kaggle/working/{model_name}'

if not os.path.exists(model_save_path):
    raise FileNotFoundError(f"The model directory {model_save_path} does not exist.")

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_save_path, trust_remote_code=True)
model = AutoModel.from_pretrained(model_save_path, trust_remote_code=True).to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at /kaggle/working/PubChem10M_SMILES_BPE_450k_model_1_pampa and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
# Load your datasets
train_df = pd.read_csv('/kaggle/input/pampa-dataset/Train.csv')
train_df = train_df[['ID', 'SMILES', 'Permeability']]
test_df = pd.read_csv('/kaggle/input/pampa-dataset/Test.csv')
test_df = test_df[['ID', 'SMILES', 'Permeability']]

In [13]:
train_encodings = tokenizer(list(train_df['SMILES']), truncation=True, padding=True, max_length=325, return_tensors="pt")
test_encodings = tokenizer(list(test_df['SMILES']), truncation=True, padding=True, max_length=325, return_tensors="pt")

In [14]:
from tqdm import tqdm 
batch_size = 16 

def generate_embeddings(encodings, batch_size):
    embeddings = []
    model.eval() 
    with torch.no_grad():
        for i in tqdm(range(0, len(encodings['input_ids']), batch_size), desc="Processing batches"):
            batch = {key: val[i:i + batch_size].to(device) for key, val in encodings.items()}  
            outputs = model(**batch)
            embeddings.append(outputs.last_hidden_state)
    return torch.cat(embeddings, dim=0)


In [15]:
train_embeddings = generate_embeddings(train_encodings, batch_size)
print(train_embeddings.shape)
train_embeddings = torch.mean(train_embeddings, dim=1)
print(train_embeddings.shape)

Processing batches: 100%|██████████| 348/348 [00:51<00:00,  6.74it/s]


torch.Size([5568, 251, 768])
torch.Size([5568, 768])


In [16]:
column_names = [f'x_fine_emb_pubchem{i}' for i in range(train_embeddings.shape[1])]
embeddings_df = pd.DataFrame(data=train_embeddings.cpu().numpy(), columns=column_names)
train_data = pd.concat([train_df, embeddings_df], axis=1)

In [17]:
test_embeddings = generate_embeddings(test_encodings, batch_size)
print(test_embeddings.shape)
test_embeddings = torch.mean(test_embeddings, dim=1)
print(test_embeddings.shape)

Processing batches: 100%|██████████| 87/87 [00:11<00:00,  7.30it/s]

torch.Size([1392, 244, 768])
torch.Size([1392, 768])


In [18]:
column_names = [f'x_fine_emb_pubchem{i}' for i in range(test_embeddings.shape[1])]
embeddings_df = pd.DataFrame(data=test_embeddings.cpu().numpy(), columns=column_names)
test_data = pd.concat([test_df, embeddings_df], axis=1)

In [19]:
train_data.to_csv("/kaggle/working/Train_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_pampa.csv",index=False)
test_data.to_csv("/kaggle/working/Test_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_pampa.csv",index=False)

In [20]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression  # LogisticRegression is not used for regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [21]:
train_data = pd.read_csv("/kaggle/working/Train_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_pampa.csv")
test_data = pd.read_csv("/kaggle/working/Test_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_pampa.csv")

In [22]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []

        

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -3.9)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -3.9)
            test_predictions_folds.append(predictions_test_fold)


        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)


        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        
        

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df



In [23]:
X_train = train_data.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_data['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

X_test = test_data.drop(['ID','SMILES','Permeability'],axis=1)
y_test = test_data['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=42),
    DecisionTreeRegressor(random_state=42),
    RandomForestRegressor(n_jobs=-1, random_state=42),
    GradientBoostingRegressor(random_state=42),
    AdaBoostRegressor(random_state=42),
    xgb.XGBRegressor(random_state=42),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=42),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=42)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 768)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 768)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.027491 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 768
[LightGBM] [Info] Start training from score -5.750121
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.027152 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 768
[LightGBM] [Info] Start training from score -5.747444
[LightGBM] [Info] Auto-choosing col-wise multi-threading

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.0848,0.2142,0.2912,0.8638,0.9294,0.9136,0.2658,0.3649,0.5155,0.5819,0.7685,0.7512
DecisionTreeRegressor,0.1870,0.3141,0.4324,0.6997,0.8517,0.8309,0.2861,0.3849,0.5349,0.5500,0.7540,0.7280
RandomForestRegressor,0.0858,0.2147,0.2930,0.8621,0.9285,0.9115,0.2635,0.3625,0.5134,0.5855,0.7706,0.7536
GradientBoostingRegressor,0.0839,0.2137,0.2897,0.8652,0.9302,0.9137,0.2690,0.3667,0.5186,0.5769,0.7663,0.7484
AdaBoostRegressor,0.1028,0.2468,0.3206,0.8350,0.9167,0.8970,0.2687,0.3788,0.5183,0.5774,0.7628,0.7419
XGBRegressor,0.1002,0.2324,0.3165,0.8391,0.9165,0.8986,0.2633,0.3650,0.5132,0.5858,0.7717,0.7538
ExtraTreesRegressor,0.0889,0.2168,0.2981,0.8573,0.9259,0.9098,0.2623,0.3623,0.5121,0.5875,0.7716,0.7540
LinearRegression,0.0936,0.2292,0.3060,0.8496,0.9223,0.9068,0.2672,0.3646,0.5169,0.5798,0.7696,0.7532
KNeighborsRegressor,0.1150,0.2482,0.3392,0.8153,0.9040,0.8793,0.2684,0.3645,0.5180,0.5779,0.7696,0.7541
SVR,0.0836,0.2117,0.2891,0.8657,0.9305,0.9153,0.2621,0.3612,0.5120,0.5877,0.7723,0.7572


In [24]:
result_df

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.0848,0.2142,0.2912,0.8638,0.9294,0.9136,0.2658,0.3649,0.5155,0.5819,0.7685,0.7512
DecisionTreeRegressor,0.1870,0.3141,0.4324,0.6997,0.8517,0.8309,0.2861,0.3849,0.5349,0.5500,0.7540,0.7280
RandomForestRegressor,0.0858,0.2147,0.2930,0.8621,0.9285,0.9115,0.2635,0.3625,0.5134,0.5855,0.7706,0.7536
GradientBoostingRegressor,0.0839,0.2137,0.2897,0.8652,0.9302,0.9137,0.2690,0.3667,0.5186,0.5769,0.7663,0.7484
AdaBoostRegressor,0.1028,0.2468,0.3206,0.8350,0.9167,0.8970,0.2687,0.3788,0.5183,0.5774,0.7628,0.7419
XGBRegressor,0.1002,0.2324,0.3165,0.8391,0.9165,0.8986,0.2633,0.3650,0.5132,0.5858,0.7717,0.7538
ExtraTreesRegressor,0.0889,0.2168,0.2981,0.8573,0.9259,0.9098,0.2623,0.3623,0.5121,0.5875,0.7716,0.7540
LinearRegression,0.0936,0.2292,0.3060,0.8496,0.9223,0.9068,0.2672,0.3646,0.5169,0.5798,0.7696,0.7532
KNeighborsRegressor,0.1150,0.2482,0.3392,0.8153,0.9040,0.8793,0.2684,0.3645,0.5180,0.5779,0.7696,0.7541
SVR,0.0836,0.2117,0.2891,0.8657,0.9305,0.9153,0.2621,0.3612,0.5120,0.5877,0.7723,0.7572


In [25]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.939915325744976, -6.882703123018174, -6.85...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.87137068190934, -6.720824193247839, -6.66...","[-5.876023420002264, -6.678260912432677, -6.64...","[0.0777259952139959, 0.0760950912927324, 0.043..."
1,DecisionTreeRegressor,"[-7.0, -6.24, -7.0, -7.0, -7.0, -7.0, -6.2, -6...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.18, -6.24, -6.05, -6.13, -4.96, -5.16, -6...","[-5.916, -6.416000000000001, -6.728219153, -6....","[0.3634611396009208, 0.24088171371027708, 0.35..."
2,RandomForestRegressor,"[-6.989696390470003, -6.898554206125, -6.92775...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.989598991729999, -6.499957021920004, -6.6...","[-5.955505912813333, -6.633371621389003, -6.64...","[0.041249224903945184, 0.11786336251683288, 0...."
3,GradientBoostingRegressor,"[-6.927879506611598, -6.799837354754541, -6.87...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.83470508853762, -6.711296618554727, -6.67...","[-5.832513268207874, -6.648379409348452, -6.62...","[0.044578294266328554, 0.049149109746613934, 0..."
4,AdaBoostRegressor,"[-6.9558179842533905, -6.9558179842533905, -6....",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.018016037382823, -6.642791736495444, -6.6...","[-5.904539631778101, -6.589483855324564, -6.65...","[0.06988241799474965, 0.14700985108970857, 0.0..."
5,XGBRegressor,"[-7.030985, -6.8406615, -7.0204477, -6.905035,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.052514, -6.333399, -6.6953754, -6.091042,...","[-5.862749, -6.528627, -6.7407675, -6.0584493,...","[0.13427448, 0.16843286, 0.101112835, 0.088937..."
6,ExtraTreesRegressor,"[-6.961824105029999, -6.870699213629999, -6.91...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.754654463724998, -6.392900000000005, -6.6...","[-5.797983536306001, -6.512091108038004, -6.63...","[0.048182088706062665, 0.16187591750188196, 0...."
7,LinearRegression,"[-6.8982207833096805, -6.852697718358306, -6.6...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.884253333199445, -6.589550395425246, -6.6...","[-5.808511503306879, -6.55635573775313, -6.470...","[0.07952436186700947, 0.14913217569117332, 0.1..."
8,KNeighborsRegressor,"[-7.0, -6.71, -6.95, -7.0, -7.0, -7.0, -5.6233...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.62, -6.746666666666667, -6.74666666666666...","[-5.676, -6.797333333333333, -6.75733333333333...","[0.05478848216347781, 0.10133333333333319, 0.0..."
9,SVR,"[-6.889767102055849, -6.807459372630064, -6.90...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.103830311880387, -6.477767688805172, -6.6...","[-5.9756816920712374, -6.580975943249352, -6.7...","[0.06936870486684368, 0.12481944576101142, 0.0..."


In [26]:
result_df.to_csv('/kaggle/working/Results_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_pampa.csv')
prediction_df.to_csv('/kaggle/working/Prediction_data_PubChem10M_SMILES_BPE_450k_model_1_fine_tuned_embeddings_pampa.csv')